# Import our functions to test

In [1]:
import diffolars.diff as dd
from diffolars.demo import get_df_pair
import polars as pl

# Testing the core usage here:

In [2]:
# Example Usage:

# Get example original and mutated dataframe pair.
# max number of columns is current 32 for this version.
dfs = get_df_pair(100, 32, n_new_cols=5, n_new_rows=5, coverage=0.5)
o = dfs['original']
m = dfs['mutated']


prune_record = dd.report_prune(o, m)

pruned_df = dd.pruned_rows(o, m)

result_df = dd.bitdiff(o, m)

print("\n")
print(pl.DataFrame(prune_record))
print(pruned_df)
result_df

Generating initial dataset with 100 rows and 32 columns.
Generated mutated dataset.
Found 0 rows unique to the current (original) table.
Found 5 rows unique to the next (latest) table.


shape: (1, 5)
┌───────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┐
│ date_pruned   ┆ cols_only_in_prev_ ┆ num_rows_only_in_p ┆ cols_only_in_lates ┆ num_rows_only_in_ │
│ ---           ┆ load               ┆ rev_load           ┆ t_load             ┆ latest_load       │
│ datetime[μs]  ┆ ---                ┆ ---                ┆ ---                ┆ ---               │
│               ┆ str                ┆ i64                ┆ str                ┆ i64               │
╞═══════════════╪════════════════════╪════════════════════╪════════════════════╪═══════════════════╡
│ 2026-07-04    ┆ No exclusives      ┆ 0                  ┆ col_35_datetime ,  ┆ 5                 │
│ 10:02:00      ┆                    ┆                    ┆ col_32_int ,…      ┆            

record_id,diff_bitarray
str,u64
"""9b0cef13-405e-47b1-95df-36051d…",2066010677
"""6feb221d-dad5-4f54-a650-cdd861…",3110807832
"""c6a81e57-ce55-4d18-8250-0531db…",1885477592
"""e5f3ca39-5066-47f9-8792-5d818d…",1370373185
"""b4a6586f-4922-4c7b-bbdc-89a552…",3065273730
…,…
"""ef909093-5b4f-4f56-ad54-02ca3e…",3566199400
"""4ec3a18b-72d7-4353-a8e3-a00475…",3762869321
"""d5ee5470-161c-4119-ac4a-055b6c…",2071337433


# Test with flat files

In [3]:
import os
from pathlib import Path
from datetime import date



scan = True
write = True
prev_load = 'original.parquet'
latest_load = 'mutated.parquet'
id_col = 'record_id'





if scan:
    o = pl.scan_parquet(prev_load)
    m = pl.scan_parquet(latest_load)
else:
    o = pl.scan_parquet(prev_load)
    m = pl.scan_parquet(latest_load)


###########################
# Generate the report log
###########################

report_prune = dd.report_prune(o, m)
report_date = report_prune['date_pruned']
report_prune |= {
    'current_load_table' : prev_load,
    'latest_load_table' : latest_load
}
report_prune_df = pl.from_dict(report_prune)
# print(report_date) # DEBUG


######################################################
# Generate the table containing record differences
######################################################
exclusive_records = dd.pruned_rows(o, m)
exclusive_records = (
    exclusive_records.with_columns(
        pl.when(pl.col('source_dataload') == 'latest load')
        .then(pl.lit(latest_load))
        .when(pl.col('source_dataload') == 'previous load')
        .then(pl.lit(prev_load))
        .otherwise(pl.lit('source table not found'))
        .alias('source_table_name')
    ).select(['date_pruned', 'source_dataload', 'source_table_name', id_col])


)

# print(exclusive_records) # DEBUG PRINT

######################################################
# Generate the table containing record differences
######################################################
bitdiff_df = dd.bitdiff(o, m)
bitdiff_df = (
    bitdiff_df.with_columns(
        pl.lit(report_date).alias('date_diffed'),
    )
).select(pl.col(['date_diffed', id_col, 'diff_bitarray']))
# print(bitdiff_df) # DEBUG PRINT

# Print to show results:
print('\n')
print("#" * 50)
print("DIFF ACTIVITY LOG RECORD\n")
print(report_prune_df)

print('\n')
print("#" * 50)
print("DIFF RECORD DIFFERENCES\n")
print(exclusive_records)

print('\n')
print("#" * 50)
print("DIFF BITARRAY RESULTS\n")
print(bitdiff_df)

try:
    
    if write:
        # The results are simply a concatenation of the original input data tables
        result_name_header = prev_load.split('.')[0] + '-' + latest_load.split('.')[0]
        
        data_path = Path('data', result_name_header, str(date.today()))

        print(f"Creating directory for results at {data_path}")
        
        os.makedirs(data_path, exist_ok=True)

        print("Writing results to flat files...")
        
        report_prune_path = data_path / 'diff_activity_log_record.parquet'
        exclusive_records_path = data_path / 'diff_record_differences.parquet'
        bitdiff_df_path = data_path / 'diff_bitarray_results.parquet'

        report_prune_df.write_parquet(report_prune_path, compression_level=20)
        exclusive_records.write_parquet(exclusive_records_path, compression_level=20)
        bitdiff_df.write_parquet(bitdiff_df_path, compression_level=20)

        print("Done!")
except Exception as e:
    print(e)



Found 0 rows unique to the current (original) table.
Found 5 rows unique to the next (latest) table.


##################################################
DIFF ACTIVITY LOG RECORD

shape: (1, 7)
┌──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ date_pruned  ┆ cols_only_i ┆ num_rows_on ┆ cols_only_i ┆ num_rows_on ┆ current_loa ┆ latest_load │
│ ---          ┆ n_prev_load ┆ ly_in_prev_ ┆ n_latest_lo ┆ ly_in_lates ┆ d_table     ┆ _table      │
│ datetime[μs] ┆ ---         ┆ load        ┆ ad          ┆ t_load      ┆ ---         ┆ ---         │
│              ┆ str         ┆ ---         ┆ ---         ┆ ---         ┆ str         ┆ str         │
│              ┆             ┆ i64         ┆ str         ┆ i64         ┆             ┆             │
╞══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ 2026-07-04   ┆ No          ┆ 0           ┆ col_35_date ┆ 5           ┆ original.pa ┆ mutated.par 

C:\Users\kenne\Documents\CSE550\GitLab\diffolars\src\diffolars\diff.py:209: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  return input.columns


In [ ]:
# CLI built here

import click


@click.command()
@click.option('--prev-load', default='original.parquet', show_default=True,
              help='Path to the previous/original data load.')
@click.option('--latest-load', default='mutated.parquet', show_default=True,
              help='Path to the latest/mutated data load.')
@click.option('--id-col', default='record_id', show_default=True,
              help='Name of the record identifier column.')
@click.option('--scan/--no-scan', default=True,
              help='Read with pl.scan_parquet (lazy) instead of pl.read_parquet (eager).')
@click.option('--write/--no-write', default=True,
              help='Write the resulting diff tables out to parquet files.')
def diff_cli(prev_load, latest_load, id_col, scan, write):
    """Diff two parquet data loads and report/write the differences."""
    if scan:
        o = pl.scan_parquet(prev_load)
        m = pl.scan_parquet(latest_load)
    else:
        o = pl.read_parquet(prev_load)
        m = pl.read_parquet(latest_load)

    ###########################
    # Generate the report log
    ###########################
    report_prune = dd.report_prune(o, m)
    report_date = report_prune['date_pruned']
    report_prune |= {
        'current_load_table': prev_load,
        'latest_load_table': latest_load
    }
    report_prune_df = pl.from_dict(report_prune)

    ######################################################
    # Generate the table containing record differences
    ######################################################
    exclusive_records = dd.pruned_rows(o, m, id_col=id_col)
    exclusive_records = (
        exclusive_records.with_columns(
            pl.when(pl.col('source_dataload') == 'latest load')
            .then(pl.lit(latest_load))
            .when(pl.col('source_dataload') == 'previous load')
            .then(pl.lit(prev_load))
            .otherwise(pl.lit('source table not found'))
            .alias('source_table_name')
        ).select(['date_pruned', 'source_dataload', 'source_table_name', id_col])
    )

    ######################################################
    # Generate the table containing bitarray differences
    ######################################################
    bitdiff_df = dd.bitdiff(o, m, id_col=id_col)
    bitdiff_df = (
        bitdiff_df.with_columns(
            pl.lit(report_date).alias('date_diffed'),
        )
    ).select(pl.col(['date_diffed', id_col, 'diff_bitarray']))

    click.echo('\n')
    click.echo('#' * 50)
    click.echo('DIFF ACTIVITY LOG RECORD\n')
    click.echo(report_prune_df)

    click.echo('\n')
    click.echo('#' * 50)
    click.echo('DIFF RECORD DIFFERENCES\n')
    click.echo(exclusive_records)

    click.echo('\n')
    click.echo('#' * 50)
    click.echo('DIFF BITARRAY RESULTS\n')
    click.echo(bitdiff_df)

    if write:
        try:
            result_name_header = Path(prev_load).stem + '-' + Path(latest_load).stem
            data_path = Path('data', result_name_header, str(date.today()))

            click.echo(f"Creating directory for results at {data_path}")
            os.makedirs(data_path, exist_ok=True)

            click.echo("Writing results to flat files...")

            report_prune_path = data_path / 'diff_activity_log_record.parquet'
            exclusive_records_path = data_path / 'diff_record_differences.parquet'
            bitdiff_df_path = data_path / 'diff_bitarray_results.parquet'

            report_prune_df.write_parquet(report_prune_path, compression_level=20)
            exclusive_records.write_parquet(exclusive_records_path, compression_level=20)
            bitdiff_df.write_parquet(bitdiff_df_path, compression_level=20)

            click.echo("Done!")
        except Exception as e:
            click.echo(e)




# Testing CLI import

This is essentially the main interface for usage.

In [25]:
from diffolars.cli import diff_cli
# In a notebook, click commands are `Command` objects, not plain functions, so
# calling `diff_cli()` directly would try to parse `sys.argv` and then `sys.exit`.
# Use `standalone_mode=False` and pass args explicitly to run it inline instead:
diff_cli(
    ['--scan', 
     '--write',
     '--prev-load', 'original.parquet', 
     '--latest-load', 'mutated.parquet', 
     '--id-col', 'record_id'],
    standalone_mode=False,
)

# From an actual terminal (e.g. after adding this as a console-script entry point),
# it would be invoked as:
#   diff-cli --prev-load original.parquet --latest-load mutated.parquet --id-col record_id



Found 0 rows unique to the current (original) table.
Found 5 rows unique to the next (latest) table.


##################################################


DIFF ACTIVITY LOG RECORD

shape: (1, 7)
┌──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ date_pruned  ┆ cols_only_i ┆ num_rows_on ┆ cols_only_i ┆ num_rows_on ┆ current_loa ┆ latest_load │
│ ---          ┆ n_prev_load ┆ ly_in_prev_ ┆ n_latest_lo ┆ ly_in_lates ┆ d_table     ┆ _table      │
│ datetime[μs] ┆ ---         ┆ load        ┆ ad          ┆ t_load      ┆ ---         ┆ ---         │
│              ┆ str         ┆ ---         ┆ ---         ┆ ---         ┆ str         ┆ str         │
│              ┆             ┆ i64         ┆ str         ┆ i64         ┆             ┆             │
╞══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ 2026-07-04   ┆ No          ┆ 0           ┆ col_35_date ┆ 5           ┆ original.pa ┆ mutated.par │
│ 10:04:00     ┆ exclusives  ┆             ┆ time ,      ┆             ┆ rquet       ┆ quet        │
│              ┆             ┆             ┆ col_32

C:\Users\kenne\Documents\CSE550\GitLab\diffolars\src\diffolars\diff.py:209: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  return input.columns


In [6]:
print(bin(199))
dd.brian_kernighan(199)

0b11000111


5

# Checking other functionalities by themselves...

In [7]:
# Here, we get the core tables:
oc, mc = dd.get_core(o, m)
print(oc.head(5))
print(mc.head(5))

shape: (5, 33)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ record_id ┆ col_0_int ┆ col_1_str ┆ col_10_fl ┆ … ┆ col_6_flo ┆ col_7_dat ┆ col_8_int ┆ col_9_st │
│ _A        ┆ _A        ┆ _A        ┆ oat_A     ┆   ┆ at_A      ┆ etime_A   ┆ _A        ┆ r_A      │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ str       ┆ i64       ┆ str       ┆ f64       ┆   ┆ f64       ┆ datetime[ ┆ i64       ┆ str      │
│           ┆           ┆           ┆           ┆   ┆           ┆ μs]       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ ac2c208b- ┆ -216755   ┆ IkuJEhoRy ┆ -223781.7 ┆ … ┆ 399391.34 ┆ 2017-08-2 ┆ -125806   ┆ ulurYeua │
│ 14c1-4170 ┆           ┆ F         ┆ 0087      ┆   ┆ 6977      ┆ 8         ┆           ┆ ue       │
│ -bb3a-766 ┆           ┆           ┆           ┆   ┆           ┆ 00:07:34  

C:\Users\kenne\Documents\CSE550\GitLab\diffolars\src\diffolars\diff.py:209: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  return input.columns


In [8]:
dd.column_intercept(o, m)

{'col_0_int',
 'col_10_float',
 'col_11_datetime',
 'col_12_int',
 'col_13_str',
 'col_14_float',
 'col_15_datetime',
 'col_16_int',
 'col_17_str',
 'col_18_float',
 'col_19_datetime',
 'col_1_str',
 'col_20_int',
 'col_21_str',
 'col_22_float',
 'col_23_datetime',
 'col_24_int',
 'col_25_str',
 'col_26_float',
 'col_27_datetime',
 'col_28_int',
 'col_29_str',
 'col_2_float',
 'col_30_float',
 'col_31_datetime',
 'col_3_datetime',
 'col_4_int',
 'col_5_str',
 'col_6_float',
 'col_7_datetime',
 'col_8_int',
 'col_9_str',
 'record_id'}

In [9]:
dd.report_prune(o, m)

{'date_pruned': datetime.datetime(2026, 7, 4, 10, 2),
 'cols_only_in_prev_load': 'No exclusives',
 'num_rows_only_in_prev_load': 0,
 'cols_only_in_latest_load': 'col_35_datetime , col_32_int , col_34_float , col_36_int , col_33_str',
 'num_rows_only_in_latest_load': 5}

In [10]:
dd.pruned_rows(o, m)

Found 0 rows unique to the current (original) table.
Found 5 rows unique to the next (latest) table.


date_pruned,source_dataload,record_id
datetime[μs],str,str
2026-07-04 10:02:00,"""latest load""","""5ebb0bfb-7de2-4e7f-9165-b08bb0…"
2026-07-04 10:02:00,"""latest load""","""ae7cb6fb-98d9-48eb-8fbd-4a6ed4…"
2026-07-04 10:02:00,"""latest load""","""0f56c470-a3f5-4ad3-a0c1-d8925a…"
2026-07-04 10:02:00,"""latest load""","""af2cc485-bcdf-4b70-9b3b-b7b372…"
2026-07-04 10:02:00,"""latest load""","""9d294ef3-fd6b-4c6b-961e-5c9f25…"


In [11]:
csd=dd.column_symmetric_diff(o, m)
csd



{'prev_load': set(),
 'latest_load': {'col_32_int',
  'col_33_str',
  'col_34_float',
  'col_35_datetime',
  'col_36_int'}}

In [12]:
dd.row_intercept(o, m)

{'0071fa6e-85df-4868-bd83-adea7ff94207',
 '062504cb-b72d-410a-a447-38a5e6b21d15',
 '081110a1-7f47-42c2-8eb3-1783e4731c58',
 '09986cc8-5fa8-4a2d-b8dc-bfcba42346f6',
 '0a5d371f-d6f8-46e6-839e-6c08f56a16d8',
 '0b1a8a7d-e7da-407b-8488-83748d87b860',
 '16874c96-04b7-4465-8b9a-e4a545e47103',
 '1c09c9fc-1259-413d-8c4d-406e49423213',
 '1c449d5e-18c3-4a3e-9cf6-a3360006255e',
 '1c521900-12f8-4baa-8e4d-f727a68f102d',
 '1cbd15e6-0e0a-49e8-ba5a-ef368d3b20a2',
 '1eb258da-8188-411d-894e-1ae18d7ff732',
 '1f62b8ee-9269-4d49-b4cb-be7cb847a2be',
 '22e20f7d-6ac8-4f74-a774-ebb014487d46',
 '23406a70-ed44-4701-b73a-ef8b05878f66',
 '2450f675-f067-4abb-80b6-8da719221daf',
 '298c957c-dbe2-45a0-acd2-7911bbd1b4c3',
 '2d6a0a18-9a25-4bda-a684-30781c0da6ad',
 '31d38818-5c7c-429a-bf18-f33458cdc6c8',
 '377179f2-b1a5-4c2b-a439-38c478689096',
 '377f1572-e507-42c2-9611-989830bfe133',
 '39002ce8-2f3d-437d-b9fa-f7176a2d579e',
 '3c18f8dd-65a6-4e92-89ad-080b0fcd8ffb',
 '3d2f6c99-bf24-4987-8ab3-23ef52a770de',
 '3f87b9d1-89c2-

In [13]:
dd.row_symmetric_diff(o, m)

{'prev_load': set(),
 'latest_load': {'0f56c470-a3f5-4ad3-a0c1-d8925a01b43b',
  '5ebb0bfb-7de2-4e7f-9165-b08bb042224c',
  '9d294ef3-fd6b-4c6b-961e-5c9f25270059',
  'ae7cb6fb-98d9-48eb-8fbd-4a6ed41ec9a0',
  'af2cc485-bcdf-4b70-9b3b-b7b3727a6552'}}

In [14]:
dd.get_cols(o)

['record_id',
 'col_0_int',
 'col_1_str',
 'col_2_float',
 'col_3_datetime',
 'col_4_int',
 'col_5_str',
 'col_6_float',
 'col_7_datetime',
 'col_8_int',
 'col_9_str',
 'col_10_float',
 'col_11_datetime',
 'col_12_int',
 'col_13_str',
 'col_14_float',
 'col_15_datetime',
 'col_16_int',
 'col_17_str',
 'col_18_float',
 'col_19_datetime',
 'col_20_int',
 'col_21_str',
 'col_22_float',
 'col_23_datetime',
 'col_24_int',
 'col_25_str',
 'col_26_float',
 'col_27_datetime',
 'col_28_int',
 'col_29_str',
 'col_30_float',
 'col_31_datetime']

In [15]:
import itertools
cols = []
for a, b in itertools.product(range(1, 5, 1), ['a', 'b']):
    cols.append(str(a) + b)

cols

['1a', '1b', '2a', '2b', '3a', '3b', '4a', '4b']

In [16]:
# Quick test for the compute_bitarray.
# We expect to get 0b1010
row = {
    'record_id_a': 'abc',
    'record_id_b': 'abc',  
    
    # 1010
    '1a': 1, 
    '2a': 0, 
    '3a': 0, 
    '4a': 0,

    # 1111 
    '1b': 1, 
    '2b': 1, 
    '3b': 1, 
    '4b':1 
}

bits = (len(row) - 2) // 2
print(bits)

# comparison should give: 1010
# expect: 0b0101
ba = dd.compute_bitarray(row)
print(bin(ba))

num_ones =  dd.brian_kernighan(ba)
print(num_ones)

num_zeros = dd.count_unpadded_zero_bits(ba)
print(num_zeros) # gives 1

print(dd.count_unpadded_zero_bits(4)) # should give 2

# So, the actual way we want to get the number of 1's and 0's 
# in our computed bit array:
print(f"This is our bitarray: {bin(ba)}")

print(f"{dd.brian_kernighan(ba)} column(s) agreed.")
print(f"{dd.count_zero_bits(ba, dd.row2num_bits(row))} column(s) differed.")


4
0b1
1
0
2
This is our bitarray: 0b1
1 column(s) agreed.
3 column(s) differed.


#### Quick check on bit negation

In [17]:
b = 0b11101

print(bin(1 << 5))


0b100000


In [18]:
int.bit_length(1 << 5)
bin((1 << 5 ) - 1) 

'0b11111'

In [19]:
~b & ((1 << 5 ) - 1) 

2

In [20]:
bin(2)

'0b10'